In [58]:
import sys
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModel
from datasets import load_dataset, Dataset, load_from_disk
from tqdm import tqdm
import sqlparse
import re

In [ ]:
dataset_path = "../data"
reloaded = load_from_disk(dataset_path)


train_ds = reloaded["train"]

Example of parsed token instance 

In [53]:

for idx, instance in tqdm(enumerate(train_ds), total=len(train_ds)):
    sql_stmt = instance['sql']
    parsed = sqlparse.parse(sql_stmt)[0]
    print(idx)
    print(parsed)
    keywords = [token.value.upper() for token in parsed.tokens if token.ttype in sqlparse.tokens.Keyword]
    for token in parsed.tokens:
        print(f"Token: {token.value} ({token.ttype})")
    print(keywords)
    break

  0%|          | 0/55409 [00:00<?, ?it/s]

0
SELECT mining_sites.name, role, COUNT(*) FROM mining_sites INNER JOIN employees ON mining_sites.id = employees.site_id GROUP BY mining_sites.name, role;
Token: SELECT (Token.Keyword.DML)
Token:   (Token.Text.Whitespace)
Token: mining_sites.name, role, COUNT(*) (None)
Token:   (Token.Text.Whitespace)
Token: FROM (Token.Keyword)
Token:   (Token.Text.Whitespace)
Token: mining_sites (None)
Token:   (Token.Text.Whitespace)
Token: INNER JOIN (Token.Keyword)
Token:   (Token.Text.Whitespace)
Token: employees (None)
Token:   (Token.Text.Whitespace)
Token: ON (Token.Keyword)
Token:   (Token.Text.Whitespace)
Token: mining_sites.id = employees.site_id (None)
Token:   (Token.Text.Whitespace)
Token: GROUP BY (Token.Keyword)
Token:   (Token.Text.Whitespace)
Token: mining_sites.name, role (None)
Token: ; (Token.Punctuation)
['SELECT', 'FROM', 'INNER JOIN', 'ON', 'GROUP BY']


Let's grab all the special tokens from this dataset

In [70]:
special_tokens = set()
for idx, instance in tqdm(enumerate(train_ds), total=len(train_ds)):
    sql_stmt = instance['sql']
    parsed = sqlparse.parse(sql_stmt)[0]
    keywords = set(token.value for token in parsed.tokens if token.ttype in sqlparse.tokens.Keyword)
    special_tokens = special_tokens | keywords


100%|██████████| 55409/55409 [00:44<00:00, 1235.19it/s]


In [71]:
special_tokens = set(filter(lambda x: x.isupper(), special_tokens))

Next, before inference, we should grab any schema related information so we have columns to check against

In [72]:
context_tokens = set()

id_ex = 5225
print(train_ds[id_ex]['sql'])
print(train_ds[id_ex]['sql_context'])

match = re.findall(r'(?<=INSERT INTO).*?\((.*?)\)', train_ds[id_ex]['sql_context'])
for grp in match:
    columns = set(grp.split(', '))
    context_tokens = context_tokens | columns


SELECT drug, phase, time AS approval_time FROM drug_approval_times GROUP BY drug, phase ORDER BY drug, phase;
CREATE TABLE drug_approval_times (drug VARCHAR(20), phase INT, time INT); INSERT INTO drug_approval_times (drug, phase, time) VALUES ('DrugX', 1, 180), ('DrugX', 2, 240), ('DrugX', 3, 300), ('DrugY', 1, 210), ('DrugY', 2, 270), ('DrugY', 3, 330);


Finally, we have the instance context and the SQL valid keywords

In [73]:
print(f"SQL keywords:{special_tokens}")
print(f"instance {id_ex} keywords: {context_tokens}")

SQL keywords:{'FULL OUTER JOIN', 'OR', 'EXCEPT', 'OF', 'DISTINCT', 'SELECT', 'FALSE', 'ASC', 'DESC', 'USING', 'YEAR', 'INTO', 'BETWEEN', 'OFFSET', 'INTERSECT', 'INSERT', 'JOIN', 'LEFT JOIN', 'UNION', 'LIMIT', 'AND', 'INNER JOIN', 'RIGHT JOIN', 'TRUE', 'LEFT OUTER JOIN', 'AS', 'GROUP BY', 'IN', 'WINDOW', 'CROSS JOIN', 'ON', 'ORDER BY', 'RETURNING', 'HAVING', 'FROM', 'UNION ALL'}
instance 5225 keywords: {'drug', 'phase', 'time'}
